# Feature Engineering

In [3]:
import pandas as pd
import numpy as np
import joblib
from sklearn.model_selection import StratifiedKFold

In [5]:
import joblib
X_train = joblib.load("../models/X_train_raw.pkl")
X_test = joblib.load("../models/X_test_raw.pkl")
y_train = joblib.load("../models/y_train.pkl")
y_test = joblib.load("../models/y_test.pkl")

In [7]:
print("X_train:", X_train.shape)
print("X_test :", X_test.shape)
print("y_train:", y_train.shape)
print("y_test :", y_test.shape)

X_train: (5634, 19)
X_test : (1409, 19)
y_train: (5634,)
y_test : (1409,)


In [8]:
print(X_train.head())

      gender  SeniorCitizen Partner Dependents  tenure PhoneService  \
3738    Male              0      No         No      35           No   
3151    Male              0     Yes        Yes      15          Yes   
4860    Male              0     Yes        Yes      13           No   
3867  Female              0     Yes         No      26          Yes   
3810    Male              0     Yes        Yes       1          Yes   

         MultipleLines InternetService OnlineSecurity OnlineBackup  \
3738  No phone service             DSL             No           No   
3151                No     Fiber optic            Yes           No   
4860  No phone service             DSL            Yes          Yes   
3867                No             DSL             No          Yes   
3810                No             DSL             No           No   

     DeviceProtection TechSupport StreamingTV StreamingMovies        Contract  \
3738              Yes          No         Yes             Yes  Month-to

#### Create tenure features

In [9]:
X_train["TenureGroup"] = pd.cut(
    X_train["tenure"],
    bins=[-1, 12, 48, 72],
    labels=["New", "Medium", "Long-term"]
)
X_test["TenureGroup"] = pd.cut(
    X_test["tenure"],
    bins=[-1, 12, 48, 72],
    labels=["New", "Medium", "Long-term"]
)

In [10]:
X_train["NewCustomer"] = (X_train["tenure"] <= 12).astype(int)
X_test["NewCustomer"]=(X_test["tenure"] <= 12).astype(int)

In [11]:
X_train["LongTermCustomer"] = (X_train["tenure"] >= 48).astype(int)
X_test["LongTermCustomer"] = (X_test["tenure"] >= 48).astype(int)

In [12]:
print(X_train[["tenure", "TenureGroup", "NewCustomer", "LongTermCustomer"]].head())

      tenure TenureGroup  NewCustomer  LongTermCustomer
3738      35      Medium            0                 0
3151      15      Medium            0                 0
4860      13      Medium            0                 0
3867      26      Medium            0                 0
3810       1         New            1                 0


#### Service features

In [13]:
X_train["ServiceCount"] = (
    X_train["PhoneService"].eq("Yes").astype(int)
    + X_train["MultipleLines"].eq("Yes").astype(int)
    + (X_train["InternetService"] != "No").astype(int)
    + X_train["OnlineSecurity"].eq("Yes").astype(int)
    + X_train["OnlineBackup"].eq("Yes").astype(int)
    + X_train["DeviceProtection"].eq("Yes").astype(int)
    + X_train["TechSupport"].eq("Yes").astype(int)
    + X_train["StreamingTV"].eq("Yes").astype(int)
    + X_train["StreamingMovies"].eq("Yes").astype(int)
)
X_test["ServiceCount"] = (
    X_test["PhoneService"].eq("Yes").astype(int)
    + X_test["MultipleLines"].eq("Yes").astype(int)
    + (X_test["InternetService"] != "No").astype(int)
    + X_test["OnlineSecurity"].eq("Yes").astype(int)
    + X_test["OnlineBackup"].eq("Yes").astype(int)
    + X_test["DeviceProtection"].eq("Yes").astype(int)
    + X_test["TechSupport"].eq("Yes").astype(int)
    + X_test["StreamingTV"].eq("Yes").astype(int)
    + X_test["StreamingMovies"].eq("Yes").astype(int)
)

In [14]:
X_train["ServiceCount"].value_counts().sort_index()

ServiceCount
1    998
2    704
3    645
4    777
5    741
6    724
7    552
8    319
9    174
Name: count, dtype: int64

#### Number of optional services

In [15]:
optional_services = [
    "MultipleLines",
    "OnlineSecurity",
    "OnlineBackup",
    "DeviceProtection",
    "TechSupport",
    "StreamingTV",
    "StreamingMovies"
]

In [16]:
X_train["OptionalServiceCount"] = (X_train[optional_services].eq("Yes").sum(axis=1))
X_test["OptionalServiceCount"] = (X_test[optional_services].eq("Yes").sum(axis=1))

In [17]:
print(X_train[["ServiceCount", "OptionalServiceCount"]].head())

      ServiceCount  OptionalServiceCount
3738             4                     3
3151             3                     1
4860             4                     3
3867             6                     4
3810             2                     0


#### Internet-service indicators

In [18]:
X_train["HasInternet"] = (X_train["InternetService"] != "No").astype(int)
X_test["HasInternet"] = (X_test["InternetService"] != "No").astype(int)

In [19]:
X_train["FiberOptic"] = (X_train["InternetService"] == "Fiber optic").astype(int)
X_test["FiberOptic"] = (X_test["InternetService"] == "Fiber optic").astype(int)

#### Billing features

In [20]:
print(X_train["TotalCharges"].dtype)

float64


In [21]:
X_train["ChargeToTenure"] = np.where(
    X_train["tenure"] > 0,
    X_train["TotalCharges"] / X_train["tenure"],
    X_train["MonthlyCharges"]
)
X_test["ChargeToTenure"] = np.where(
    X_test["tenure"] > 0,
    X_test["TotalCharges"] / X_test["tenure"],
    X_test["MonthlyCharges"]
)

#### Monthly charge groups

In [22]:
charge_bins = [-np.inf, 40, 80, np.inf]
charge_labels = ["Low", "Medium", "High"]
X_train["MonthlyChargeGroup"] = pd.cut(
    X_train["MonthlyCharges"],
    bins=charge_bins,
    labels=charge_labels
)
X_test["MonthlyChargeGroup"] = pd.cut(
    X_test["MonthlyCharges"],
    bins=charge_bins,
    labels=charge_labels
)

#### Total charge groups

In [23]:
total_charge_bins = [-np.inf, 1000, 5000, np.inf]
X_train["TotalChargeGroup"] = pd.cut(
    X_train["TotalCharges"],
    bins=total_charge_bins,
    labels=charge_labels
)
X_test["TotalChargeGroup"] = pd.cut(
    X_test["TotalCharges"],
    bins=total_charge_bins,
    labels=charge_labels
)

#### Service combination

In [24]:
X_train["ServiceCombination"] = (
    X_train["InternetService"].astype(str)
    + "_"
    + X_train["Contract"].astype(str)
)

X_test["ServiceCombination"] = (
    X_test["InternetService"].astype(str)
    + "_"
    + X_test["Contract"].astype(str)
)

In [25]:
X_train["ServiceCombination"].value_counts()

ServiceCombination
Fiber optic_Month-to-month    1707
DSL_Month-to-month             973
DSL_Two year                   512
No_Two year                    507
DSL_One year                   452
Fiber optic_One year           436
No_Month-to-month              422
Fiber optic_Two year           340
No_One year                    285
Name: count, dtype: int64

In [26]:
print("Train shape:", X_train.shape)
print("Test shape :", X_test.shape)
print("\nNew columns:")
print([
    col for col in X_train.columns
    if col in [
        "TenureGroup",
        "NewCustomer",
        "LongTermCustomer",
        "ServiceCount",
        "OptionalServiceCount",
        "HasInternet",
        "FiberOptic",
        "ChargeToTenure",
        "MonthlyChargeGroup",
        "TotalChargeGroup",
        "ServiceCombination"
    ]
])

Train shape: (5634, 30)
Test shape : (1409, 30)

New columns:
['TenureGroup', 'NewCustomer', 'LongTermCustomer', 'ServiceCount', 'OptionalServiceCount', 'HasInternet', 'FiberOptic', 'ChargeToTenure', 'MonthlyChargeGroup', 'TotalChargeGroup', 'ServiceCombination']


#### Create ContractRisk

In [27]:
train_contract = pd.DataFrame({
    "Contract": X_train["Contract"],
    "Churn": (y_train == "Yes").astype(int)
})
contract_risk = (
    train_contract
    .groupby("Contract")["Churn"]
    .mean()
)
contract_risk

Contract
Month-to-month    0.427466
One year          0.110827
Two year          0.028698
Name: Churn, dtype: float64

In [28]:
X_train["ContractRisk"] = (X_train["Contract"].map(contract_risk))
X_test["ContractRisk"] = (X_test["Contract"].map(contract_risk))

In [29]:
X_train[["Contract", "ContractRisk"]].head()

,Contract,ContractRisk
3738,Month-to-month,0.427466
3151,Month-to-month,0.427466
4860,Two year,0.028698
3867,Two year,0.028698
3810,Month-to-month,0.427466


#### Payment method risk

In [30]:
train_payment = pd.DataFrame({
    "PaymentMethod": X_train["PaymentMethod"],
    "Churn": (y_train == "Yes").astype(int)
})
payment_risk = (
    train_payment
    .groupby("PaymentMethod")["Churn"]
    .mean()
)
payment_risk

PaymentMethod
Bank transfer (automatic)    0.161576
Credit card (automatic)      0.149217
Electronic check             0.457430
Mailed check                 0.192846
Name: Churn, dtype: float64

In [31]:
X_train["PaymentMethodRisk"] = (X_train["PaymentMethod"].map(payment_risk))
X_test["PaymentMethodRisk"] = (X_test["PaymentMethod"].map(payment_risk))

In [32]:
X_train[["PaymentMethod", "PaymentMethodRisk"]].head()

,PaymentMethod,PaymentMethodRisk
3738,Electronic check,0.457430
3151,Mailed check,0.192846
4860,Mailed check,0.192846
3867,Credit card (automatic),0.149217
3810,Electronic check,0.457430


#### Service combination risk

In [33]:
train_service = pd.DataFrame({
    "ServiceCombination": X_train["ServiceCombination"],
    "Churn": (y_train == "Yes").astype(int)
})

service_risk = (
    train_service
    .groupby("ServiceCombination")["Churn"]
    .mean()
)

service_risk

ServiceCombination
DSL_Month-to-month            0.317575
DSL_One year                  0.095133
DSL_Two year                  0.019531
Fiber optic_Month-to-month    0.550674
Fiber optic_One year          0.185780
Fiber optic_Two year          0.070588
No_Month-to-month             0.182464
No_One year                   0.021053
No_Two year                   0.009862
Name: Churn, dtype: float64

In [34]:
X_train["ServiceCombinationRisk"] = (X_train["ServiceCombination"].map(service_risk))
X_test["ServiceCombinationRisk"] = (X_test["ServiceCombination"].map(service_risk))

#### Handle an unseen test category

In [35]:
overall_train_churn = (y_train == "Yes").mean()

In [36]:
X_train["ContractRisk"] = (
    X_train["ContractRisk"]
    .fillna(overall_train_churn)
)
X_test["ContractRisk"] = (
    X_test["ContractRisk"]
    .fillna(overall_train_churn)
)
X_train["PaymentMethodRisk"] = (
    X_train["PaymentMethodRisk"]
    .fillna(overall_train_churn)
)
X_test["PaymentMethodRisk"] = (
    X_test["PaymentMethodRisk"]
    .fillna(overall_train_churn)
)
X_train["ServiceCombinationRisk"] = (
    X_train["ServiceCombinationRisk"]
    .fillna(overall_train_churn)
)
X_test["ServiceCombinationRisk"] = (
    X_test["ServiceCombinationRisk"]
    .fillna(overall_train_churn)
)

In [37]:
print(
    X_train[
        [
            "ContractRisk",
            "PaymentMethodRisk",
            "ServiceCombinationRisk"
        ]
    ].isnull().sum()
)

print(
    X_test[
        [
            "ContractRisk",
            "PaymentMethodRisk",
            "ServiceCombinationRisk"
        ]
    ].isnull().sum()
)

ContractRisk              0
PaymentMethodRisk         0
ServiceCombinationRisk    0
dtype: int64
ContractRisk              0
PaymentMethodRisk         0
ServiceCombinationRisk    0
dtype: int64


#### High-value feature

In [38]:
high_value_threshold = X_train["TotalCharges"].quantile(0.75)
print("High-value threshold:", high_value_threshold)

High-value threshold: 3835.8250000000003


In [39]:
X_train["HighValue"] = (X_train["TotalCharges"] >= high_value_threshold).astype(int)
X_test["HighValue"] = (X_test["TotalCharges"] >= high_value_threshold).astype(int)

#### Create HighValueHighRisk

In [40]:
contract_risk_threshold = X_train["ContractRisk"].median()
print("Contract risk threshold:",contract_risk_threshold)

Contract risk threshold: 0.4274661508704062


In [41]:
X_train["HighValueHighRisk"] = (
    (X_train["HighValue"] == 1) &
    (X_train["ContractRisk"] >= contract_risk_threshold)
).astype(int)
X_test["HighValueHighRisk"] = (
    (X_test["HighValue"] == 1) &
    (X_test["ContractRisk"] >= contract_risk_threshold)
).astype(int)

#### Check all risk features

In [42]:
risk_features = [
    "ContractRisk",
    "PaymentMethodRisk",
    "ServiceCombinationRisk",
    "HighValue",
    "HighValueHighRisk"
]

In [43]:
X_train[risk_features].head()

,ContractRisk,PaymentMethodRisk,ServiceCombinationRisk,HighValue,HighValueHighRisk
3738,0.427466,0.457430,0.317575,0,0
3151,0.427466,0.192846,0.550674,0,0
4860,0.028698,0.192846,0.019531,0,0
3867,0.028698,0.149217,0.019531,0,0
3810,0.427466,0.457430,0.317575,0,0


In [44]:
print("Training missing values:")
print(X_train[risk_features].isnull().sum())

print("\nTesting missing values:")
print(X_test[risk_features].isnull().sum())

Training missing values:
ContractRisk              0
PaymentMethodRisk         0
ServiceCombinationRisk    0
HighValue                 0
HighValueHighRisk         0
dtype: int64

Testing missing values:
ContractRisk              0
PaymentMethodRisk         0
ServiceCombinationRisk    0
HighValue                 0
HighValueHighRisk         0
dtype: int64


#### Check the complete feature-engineered dataset

In [45]:
engineered_features = [
    "TenureGroup",
    "NewCustomer",
    "LongTermCustomer",
    "ServiceCount",
    "OptionalServiceCount",
    "HasInternet",
    "FiberOptic",
    "MonthlyChargeGroup",
    "TotalChargeGroup",
    "ChargeToTenure",
    "ServiceCombination",
    "ContractRisk",
    "PaymentMethodRisk",
    "ServiceCombinationRisk",
    "HighValue",
    "HighValueHighRisk"
]

In [46]:
print("Engineered features:")
print(engineered_features)

print("\nNumber of engineered features:",
      len(engineered_features))

Engineered features:
['TenureGroup', 'NewCustomer', 'LongTermCustomer', 'ServiceCount', 'OptionalServiceCount', 'HasInternet', 'FiberOptic', 'MonthlyChargeGroup', 'TotalChargeGroup', 'ChargeToTenure', 'ServiceCombination', 'ContractRisk', 'PaymentMethodRisk', 'ServiceCombinationRisk', 'HighValue', 'HighValueHighRisk']

Number of engineered features: 16


In [47]:
X_train[engineered_features].head()

,TenureGroup,NewCustomer,LongTermCustomer,ServiceCount,OptionalServiceCount,HasInternet,FiberOptic,MonthlyChargeGroup,TotalChargeGroup,ChargeToTenure,ServiceCombination,ContractRisk,PaymentMethodRisk,ServiceCombinationRisk,HighValue,HighValueHighRisk
3738,Medium,0,0,4,3,1,0,Medium,Medium,48.618571,DSL_Month-to-month,0.427466,0.457430,0.317575,0,0
3151,Medium,0,0,3,1,1,1,Medium,Medium,76.770000,Fiber optic_Month-to-month,0.427466,0.192846,0.550674,0,0
4860,Medium,0,0,4,3,1,0,Medium,Low,45.411538,DSL_Two year,0.028698,0.192846,0.019531,0,0
3867,Medium,0,0,6,4,1,0,Medium,Medium,73.296154,DSL_Two year,0.028698,0.149217,0.019531,0,0
3810,New,1,0,2,0,1,0,Medium,Low,44.550000,DSL_Month-to-month,0.427466,0.457430,0.317575,0,0


In [48]:
X_train.dtypes

gender                         str
SeniorCitizen                int64
Partner                        str
Dependents                     str
tenure                       int64
PhoneService                   str
MultipleLines                  str
InternetService                str
OnlineSecurity                 str
OnlineBackup                   str
DeviceProtection               str
TechSupport                    str
StreamingTV                    str
StreamingMovies                str
Contract                       str
PaperlessBilling               str
PaymentMethod                  str
MonthlyCharges             float64
TotalCharges               float64
TenureGroup               category
NewCustomer                  int64
LongTermCustomer             int64
ServiceCount                 int64
OptionalServiceCount         int64
HasInternet                  int64
FiberOptic                   int64
ChargeToTenure             float64
MonthlyChargeGroup        category
TotalChargeGroup    

In [49]:
numerical_features = [
    "SeniorCitizen",
    "tenure",
    "MonthlyCharges",
    "TotalCharges",
    "NewCustomer",
    "LongTermCustomer",
    "ServiceCount",
    "OptionalServiceCount",
    "HasInternet",
    "FiberOptic",
    "ChargeToTenure",
    "ContractRisk",
    "PaymentMethodRisk",
    "ServiceCombinationRisk",
    "HighValue",
    "HighValueHighRisk"
]
categorical_features = [
    "gender",
    "Partner",
    "Dependents",
    "PhoneService",
    "MultipleLines",
    "InternetService",
    "OnlineSecurity",
    "OnlineBackup",
    "DeviceProtection",
    "TechSupport",
    "StreamingTV",
    "StreamingMovies",
    "Contract",
    "PaperlessBilling",
    "PaymentMethod",
    "TenureGroup",
    "MonthlyChargeGroup",
    "TotalChargeGroup",
    "ServiceCombination"
]

In [50]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

final_preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            StandardScaler(),
            numerical_features
        ),
        (
            "cat",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False
            ),
            categorical_features
        )
    ]
)

In [51]:
X_train_final = final_preprocessor.fit_transform(X_train)

X_test_final = final_preprocessor.transform(X_test)

In [52]:
print("Original X_train:", X_train.shape)
print("Final X_train   :", X_train_final.shape)

print("Original X_test :", X_test.shape)
print("Final X_test    :", X_test_final.shape)

Original X_train: (5634, 35)
Final X_train   : (5634, 75)
Original X_test : (1409, 35)
Final X_test    : (1409, 75)


In [53]:
print("NaN in training:", np.isnan(X_train_final).sum())
print("NaN in testing :", np.isnan(X_test_final).sum())

NaN in training: 0
NaN in testing : 0


In [54]:
print("Inf in training:", np.isinf(X_train_final).sum())
print("Inf in testing :", np.isinf(X_test_final).sum())

Inf in training: 0
Inf in testing : 0


In [55]:
joblib.dump(final_preprocessor,"../models/final_preprocessor.pkl")
joblib.dump(X_train_final,"../models/X_train_engineered_processed.pkl")
joblib.dump(X_test_final,"../models/X_test_engineered_processed.pkl")
joblib.dump(y_train,"../models/y_train.pkl")
joblib.dump(y_test,"../models/y_test.pkl")

['../models/y_test.pkl']